In [1]:
import torch
import gc

gc.collect() # clear the CPU memory (RAM)
torch.cuda.empty_cache() #clear the GPU memory (VRAM)

## gc is Garbage Collector module. Python automatically manages memory by deleting objects that are no longer being used, and this module gives you manual control over that process.

I did this because I experience Out of memory error when I performed largescale generation. empty_cache() only clears unused cached memory

# This notebook contains log of image generation

In [ ]:
# FLUX with CPU offloading - uses system RAM instead of GPU VRAM
import torch
from diffusers import FluxPipeline

print("Loading FLUX.1-schnell with CPU offloading...")

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.float16, # cut the VRAM usage by 50% and
    load_in_8bit=True, # reduce the model weight down to 8-bit
)

pipe.enable_sequential_cpu_offload()  ## 

"""
Instead of cramming the entire giant model into your GPU VRAM at once, 
this command tells PyTorch to keep the whole thing in your system RAM. It will load Sub-Model A to the GPU, process the data, 
send the result back, pull Sub-Model A out of the GPU, and then push Sub-Model B in. It repeats this sequence step-by-step.
"""

print("FLUX loaded with CPU offloading!")

# Generate with smaller settings
image = pipe(
    "A rich brown leather boat shoe with tan accents, lacing, and a white sole, isolated on a seamless white background",
    num_inference_steps=8, ## How many times the AI cleans up the random noise to create a sharp image.
    guidance_scale=3.5, # How closely the AI adheres to your prompt
    height=512,  # Smaller resolution
    width=512
).images[0] ## first image

image.show()

Loading FLUX.1-schnell with CPU offloading...


Keyword arguments {'load_in_8bit': True} are not expected by FluxPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

FLUX loaded with CPU offloading!


  0%|          | 0/8 [00:00<?, ?it/s]

In [2]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0 # Tracks how many new images were successfully saved during this run
    session_batch_count = 0 #  to use for long cooling breaks> Tracks how many images have been processed in a row. This is used to trigger the long cooling breaks

    for i, row in enumerate(df.itertuples()):
        """
        .iterrows() (which turns each row into a heavy Pandas Series object), 
        .itertuples() can be 10 to 100 times faster. When you are processing thousands of image captions, 
        that speed difference saves you a lot of time
        """
        current_num = start_idx + i
        if current_num > end_idx: #the loop will stop if the current number in the loop is greater than end idex
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists(): #checks if an image already exists in  folder
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:  # if it finished generating 10 images, 
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache() # Flushes out all cached VRAM and system memory built up over the last 10 images
            gc.collect() 
            time.sleep(600) # 10 minutes
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip() 
            "Look at the current row we are on, find the column named 'gemini_caption', and grab the text inside it."
            if not caption or caption.lower() == "nan": ## if none then skip that row. 
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode(): ##Tells PyTorch: "We are just generating images, not training a model."
                #This deactivates gradient tracking, which cuts VRAM usage drastically and speeds up generation.
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, ## How many times the AI cleans up the random noise to create a sharp image.
                    guidance_scale=1.0, # How closely the AI adheres to your prompt
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            """
            If a single image fails to generate (maybe a corrupted text prompt or a minor hiccup), 
            this prevents the entire batch from crashing. It prints the error, waits 5 seconds, and moves on to the next row.
            """
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__": # if true then, the above code run
    """
    This is the main execution block in Python.
    to prevent automatic execution when the file is imported into another module.
    Run this code only if this file is being executed directly
    """
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3369_3379.csv", 
        start_idx=3370,
        end_idx=3379
    )

--- Gentle Session Started ---
Targeting: 3370 to 3379
[3370] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3371] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3372] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3378] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3379] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 56.13 minutes


In [3]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3379_3389.csv", 
        start_idx=3380,
        end_idx=3389
    )

--- Gentle Session Started ---
Targeting: 3380 to 3389
[3380] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3381] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3382] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3388] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3389] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 56.50 minutes


In [4]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3389_3399.csv", 
        start_idx=3390,
        end_idx=3399
    )

--- Gentle Session Started ---
Targeting: 3390 to 3399
[3390] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3391] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3392] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3398] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3399] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 56.88 minutes


In [5]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3399_3409.csv", 
        start_idx=3400,
        end_idx=3409
    )

--- Gentle Session Started ---
Targeting: 3400 to 3409
[3400] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3401] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3402] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3408] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3409] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 55.99 minutes


In [6]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3409_3419.csv", 
        start_idx=3410,
        end_idx=3419
    )

--- Gentle Session Started ---
Targeting: 3410 to 3419
[3410] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3411] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3412] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3418] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3419] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 54.22 minutes


In [7]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3419_3429.csv", 
        start_idx=3420,
        end_idx=3429
    )


--- Gentle Session Started ---
Targeting: 3420 to 3429
[3420] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3421] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3422] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3428] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3429] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 54.61 minutes


In [8]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3429_3439.csv", 
        start_idx=3430,
        end_idx=3439
    )


--- Gentle Session Started ---
Targeting: 3430 to 3439
[3430] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3431] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3432] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3438] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3439] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 54.27 minutes


In [9]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3439_3449.csv", 
        start_idx=3440,
        end_idx=3449
    )


--- Gentle Session Started ---
Targeting: 3440 to 3449
[3440] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3441] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3442] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3448] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3449] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 54.28 minutes


In [10]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3449_3459.csv", 
        start_idx=3450,
        end_idx=3459
    )


--- Gentle Session Started ---
Targeting: 3450 to 3459
[3450] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3451] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3452] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3458] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3459] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 56.43 minutes


In [11]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3459_3469.csv", 
        start_idx=3460,
        end_idx=3469
    )

--- Gentle Session Started ---
Targeting: 3460 to 3469
[3460] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3461] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3462] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3468] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3469] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 55.21 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3469_3479.csv", 
        start_idx=3470,
        end_idx=3479
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3479_3489.csv", 
        start_idx=3480,
        end_idx=3489
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3489_3499.csv", 
        start_idx=3490,
        end_idx=3499
    )

--- Gentle Session Started ---
Targeting: 3490 to 3499
[3491] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3492] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3493] Generating image... (Session: 3)


In [15]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3499_3509.csv", 
        start_idx=3500,
        end_idx=3509
    )

--- Gentle Session Started ---
Targeting: 3500 to 3509
[3500] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3501] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3502] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3508] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3509] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 103.05 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3509_3519.csv", 
        start_idx=3510,
        end_idx=3519
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3519_3529.csv", 
        start_idx=3520,
        end_idx=3529
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3529_3539.csv", 
        start_idx=3530,
        end_idx=3539
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3539_3549.csv", 
        start_idx=3540,
        end_idx=3549
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3549_3559.csv", 
        start_idx=3550,
        end_idx=3559
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3559_3569.csv", 
        start_idx=3560,
        end_idx=3569
    )

## Break

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3569_3579.csv", 
        start_idx=3570,
        end_idx=3579
    )

--- Gentle Session Started ---
Targeting: 3570 to 3579
[3570] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3571] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3572] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3573] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3574] Generating image... (Session: 5)


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3579_3589.csv", 
        start_idx=3580,
        end_idx=3589
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3589_3599.csv", 
        start_idx=3590,
        end_idx=3599
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3599_3609.csv", 
        start_idx=3600,
        end_idx=3609
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3609_3619.csv", 
        start_idx=3610,
        end_idx=3619
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3619_3629.csv", 
        start_idx=3620,
        end_idx=3629
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3629_3639.csv", 
        start_idx=3630,
        end_idx=3639
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3639_3649.csv", 
        start_idx=3640,
        end_idx=3649
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3649_3659.csv", 
        start_idx=3650,
        end_idx=3659
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3659_3669.csv", 
        start_idx=3660,
        end_idx=3669
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3669_3679.csv", 
        start_idx=3670,
        end_idx=3679
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3679_3689.csv", 
        start_idx=3680,
        end_idx=3689
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3689_3699.csv", 
        start_idx=3690,
        end_idx=3699
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3699_3709.csv", 
        start_idx=3700,
        end_idx=3709
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3709_3719.csv", 
        start_idx=3710,
        end_idx=3720
    )

--- Gentle Session Started ---
Targeting: 3710 to 3720
[3710] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3711] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3712] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3719_3729.csv", 
        start_idx=3720,
        end_idx=3729
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3729_3739.csv", 
        start_idx=3730,
        end_idx=3739
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3739_3749.csv", 
        start_idx=3740,
        end_idx=3749
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3749_3759.csv", 
        start_idx=3750,
        end_idx=3759
    )

--- Gentle Session Started ---
Targeting: 3750 to 3759
[3751] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3752] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3759_3769.csv", 
        start_idx=3760,
        end_idx=3769
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3769_3779.csv", 
        start_idx=3770,
        end_idx=3779
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3779_3789.csv", 
        start_idx=3780,
        end_idx=3789
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3789_3799.csv", 
        start_idx=3790,
        end_idx=3799
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3799_3809.csv", 
        start_idx=3800,
        end_idx=3809
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3809_3819.csv", 
        start_idx=3809,
        end_idx=3819
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3819_3829.csv", 
        start_idx=3820,
        end_idx=3829
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3829_3839.csv", 
        start_idx=3830,
        end_idx=3839
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3839_3849.csv", 
        start_idx=3840,
        end_idx=3849
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3849_3859.csv", 
        start_idx=3850,
        end_idx=3859
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3859_3869.csv", 
        start_idx=3860,
        end_idx=3869
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3869_3879.csv", 
        start_idx=3870,
        end_idx=3879
    )

--- Gentle Session Started ---
Targeting: 3870 to 3879
[3870] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3871] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3872] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3873] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3879_3889.csv", 
        start_idx=3880,
        end_idx=3889
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3889_3899.csv", 
        start_idx=3890,
        end_idx=3899
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3899_3909.csv", 
        start_idx=3899,
        end_idx=3909
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3909_3919.csv", 
        start_idx=3910,
        end_idx=3919
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3919_3929.csv", 
        start_idx=3920,
        end_idx=3929
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3929_3939.csv", 
        start_idx=3930,
        end_idx=3939
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3939_3949.csv", 
        start_idx=3940,
        end_idx=3949
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3949_3959.csv", 
        start_idx=3950,
        end_idx=3959
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3959_3969.csv", 
        start_idx=3960,
        end_idx=3969
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3969_3979.csv", 
        start_idx=3970,
        end_idx=3979
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3979_3989.csv", 
        start_idx=3980,
        end_idx=3989
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3989_3999.csv", 
        start_idx=3990,
        end_idx=3999
    )

## Break

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3999_4009.csv", 
        start_idx=4000,
        end_idx=4009
    )

--- Gentle Session Started ---
Targeting: 4000 to 4009
[4000] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4001] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4002] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4007] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4008] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4009] Generating image... (Session: 10)


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4029_4039.csv", 
        start_idx=4030,
        end_idx=4039
    )

--- Gentle Session Started ---
Targeting: 4030 to 4039
[4030] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4031] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4032] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4033] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4039_4049.csv", 
        start_idx=4040,
        end_idx=4049
    )

In [3]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4049_4059.csv", 
        start_idx=4050,
        end_idx=4059
    )

--- Gentle Session Started ---
Targeting: 4050 to 4059

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.01 minutes


In [2]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
         pipe=pipe, 
        csv_path="captions_4059_4069.csv", 
        start_idx=4060,
        end_idx=4069
    )

--- Gentle Session Started ---
Targeting: 4060 to 4069
[4060] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4061] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4062] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4068] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4069] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 94.85 minutes


In [3]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4069_4079.csv", 
        start_idx=4070,
        end_idx=4079
    )

--- Gentle Session Started ---
Targeting: 4070 to 4079
[4070] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4071] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4072] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4078] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4079] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 59.17 minutes


In [4]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4079_4089.csv", 
        start_idx=4080,
        end_idx=4089
    )

--- Gentle Session Started ---
Targeting: 4080 to 4089
[4080] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4081] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4082] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4088] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4089] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 58.76 minutes


In [5]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4089_4099.csv", 
        start_idx=4090,
        end_idx=4099
    )

--- Gentle Session Started ---
Targeting: 4090 to 4099
[4090] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4091] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4092] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4098] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4099] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 61.86 minutes


In [6]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4099_4109.csv", 
        start_idx=4100,
        end_idx=4109
    )

--- Gentle Session Started ---
Targeting: 4100 to 4109
[4100] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4101] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4102] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4108] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4109] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 59.39 minutes


In [7]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4109_4119.csv", 
        start_idx=4110,
        end_idx=4119
    )

--- Gentle Session Started ---
Targeting: 4110 to 4119
[4110] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4111] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4112] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4118] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4119] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 60.21 minutes


In [8]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4119_4129.csv", 
        start_idx=4120,
        end_idx=4129
    )

--- Gentle Session Started ---
Targeting: 4120 to 4129
[4120] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4121] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4122] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4128] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4129] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 61.06 minutes


In [9]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4129_4139.csv", 
        start_idx=4130,
        end_idx=4139
    )

--- Gentle Session Started ---
Targeting: 4130 to 4139
[4130] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4131] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4132] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4138] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4139] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 60.14 minutes


In [11]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4139_4149.csv", 
        start_idx=4140,
        end_idx=4149
    )

--- Gentle Session Started ---
Targeting: 4140 to 4149

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4149_4159.csv", 
        start_idx=4150,
        end_idx=4159
    )

In [12]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4159_4169.csv", 
        start_idx=4160,
        end_idx=4169
    )

--- Gentle Session Started ---
Targeting: 4160 to 4169
[4160] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4161] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4162] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4168] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4169] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 59.80 minutes


In [15]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4169_41799.csv", 
        start_idx=4170,
        end_idx=4179
    )

--- Gentle Session Started ---
Targeting: 4170 to 4179
[4170] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4171] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4172] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4178] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4179] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 68.33 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4169_4179.csv", 
        start_idx=4170,
        end_idx=4179
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4179_4189.csv", 
        start_idx=4180,
        end_idx=4189
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4189_4199.csv", 
        start_idx=4190,
        end_idx=4199
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4199_4209.csv", 
        start_idx=4200,
        end_idx=4209
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4209_4219.csv", 
        start_idx=4210,
        end_idx=4219
    )

--- Gentle Session Started ---
Targeting: 4210 to 4219
[4211] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

In [23]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4219_4229.csv", 
        start_idx=4220,
        end_idx=4229
    )

--- Gentle Session Started ---
Targeting: 4220 to 4229
[4220] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4221] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4222] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4228] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4229] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 65.06 minutes


In [24]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4229_4239.csv", 
        start_idx=4230,
        end_idx=4239
    )

--- Gentle Session Started ---
Targeting: 4230 to 4239
[4230] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4231] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4232] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4238] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4239] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 57.60 minutes


In [25]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4239_4249.csv", 
        start_idx=4240,
        end_idx=4249
    )

--- Gentle Session Started ---
Targeting: 4240 to 4249
[4240] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4241] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4242] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4248] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4249] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 59.67 minutes


In [26]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4249_4259.csv", 
        start_idx=4250,
        end_idx=4259
    )

--- Gentle Session Started ---
Targeting: 4250 to 4259
[4250] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4251] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4252] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4258] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4259] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 58.89 minutes


In [28]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4259_4269.csv", 
        start_idx=4260,
        end_idx=4269
    )

--- Gentle Session Started ---
Targeting: 4260 to 4269
[4260] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4261] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4262] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4268] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4269] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 58.40 minutes


In [29]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4269_4279.csv", 
        start_idx=4270,
        end_idx=4279
    )

--- Gentle Session Started ---
Targeting: 4270 to 4279
[4270] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4271] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4272] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4278] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4279] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 60.18 minutes


In [30]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4279_4289.csv", 
        start_idx=4280,
        end_idx=4289
    )

--- Gentle Session Started ---
Targeting: 4280 to 4289
[4280] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4281] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4282] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4288] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4289] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 56.28 minutes


In [31]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4289_4299.csv", 
        start_idx=4290,
        end_idx=4299
    )

--- Gentle Session Started ---
Targeting: 4290 to 4299
[4290] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4291] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4292] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4298] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4299] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 57.80 minutes


In [32]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4299_4309.csv", 
        start_idx=4299,
        end_idx=4309
    )

--- Gentle Session Started ---
Targeting: 4299 to 4309
[4300] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4301] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4302] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4307] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4308] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 52.38 minutes


In [33]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4309_4319.csv", 
        start_idx=4309,
        end_idx=4319
    )

--- Gentle Session Started ---
Targeting: 4309 to 4319
[4309] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4310] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4311] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4317] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4318] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 62.24 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4319_4329.csv", 
        start_idx=4319,
        end_idx=4329
    )

--- Gentle Session Started ---
Targeting: 4319 to 4329
[4319] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4329_4339.csv", 
        start_idx=4329,
        end_idx=4339
    )

In [47]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4339_4349.csv", 
        start_idx=4340,
        end_idx=4349
    )

--- Gentle Session Started ---
Targeting: 4340 to 4349

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [48]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4349_4359.csv", 
        start_idx=4350,
        end_idx=4359
    )

--- Gentle Session Started ---
Targeting: 4350 to 4359
[4359] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.06 minutes


In [49]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4359_4369.csv", 
        start_idx=4360,
        end_idx=4369
    )

--- Gentle Session Started ---
Targeting: 4360 to 4369
[4360] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4361] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4362] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4367] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4368] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 63.12 minutes


In [50]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4369_4379.csv", 
        start_idx=4370,
        end_idx=4379
    )

--- Gentle Session Started ---
Targeting: 4370 to 4379
[4379] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.66 minutes


In [51]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4379_4389.csv", 
        start_idx=4380,
        end_idx=4389
    )

--- Gentle Session Started ---
Targeting: 4380 to 4389
[4380] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4381] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4382] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4387] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4388] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 59.44 minutes


In [52]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4389_4399.csv", 
        start_idx=4390,
        end_idx=4399
    )

--- Gentle Session Started ---
Targeting: 4390 to 4399
[4399] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 9.64 minutes


In [53]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4399_4409.csv", 
        start_idx=4400,
        end_idx=4409
    )

--- Gentle Session Started ---
Targeting: 4400 to 4409
[4400] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4401] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4402] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4407] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4408] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 61.69 minutes


In [54]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4409_4419.csv", 
        start_idx=4410,
        end_idx=4419
    )

--- Gentle Session Started ---
Targeting: 4410 to 4419
[4415] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4416] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4417] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4418] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4419] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 33.94 minutes


In [55]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4419_4429.csv", 
        start_idx=4420,
        end_idx=4429
    )

--- Gentle Session Started ---
Targeting: 4420 to 4429
[4420] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4421] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4422] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4428] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4429] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 65.87 minutes


In [56]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4429_4439.csv", 
        start_idx=4430,
        end_idx=4439
    )

--- Gentle Session Started ---
Targeting: 4430 to 4439
[4430] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4431] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4432] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4438] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4439] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 64.17 minutes


In [57]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4439_4449.csv", 
        start_idx=4440,
        end_idx=4449
    )

--- Gentle Session Started ---
Targeting: 4440 to 4449
[4440] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4441] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4442] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4447] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4448] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 59.50 minutes


In [58]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4449_4459.csv", 
        start_idx=4450,
        end_idx=4459
    )

--- Gentle Session Started ---
Targeting: 4450 to 4459

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.01 minutes


In [59]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4449_4459.csv", 
        start_idx=4450,
        end_idx=4459
    )

--- Gentle Session Started ---
Targeting: 4450 to 4459

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [60]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4459_4469.csv", 
        start_idx=4460,
        end_idx=4469
    )

--- Gentle Session Started ---
Targeting: 4460 to 4469

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [61]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4469_4479.csv", 
        start_idx=4470,
        end_idx=4479
    )

--- Gentle Session Started ---
Targeting: 4470 to 4479
[4479] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.52 minutes


In [62]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4479_4489.csv", 
        start_idx=4480,
        end_idx=4489
    )

--- Gentle Session Started ---
Targeting: 4480 to 4489
[4480] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4481] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4482] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4487] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4488] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 60.52 minutes


In [63]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4489_4499.csv", 
        start_idx=4490,
        end_idx=4499
    )

--- Gentle Session Started ---
Targeting: 4490 to 4499
[4499] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.18 minutes


In [64]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4489_4499.csv", 
        start_idx=4490,
        end_idx=4499
    )

--- Gentle Session Started ---
Targeting: 4490 to 4499

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.01 minutes


In [65]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4499_4509.csv", 
        start_idx=4500,
        end_idx=4509
    )

--- Gentle Session Started ---
Targeting: 4500 to 4509
[4500] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4501] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4502] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4508] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4509] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 69.77 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4509_4519.csv", 
        start_idx=4510,
        end_idx=4519
    )

--- Gentle Session Started ---
Targeting: 4510 to 4519
[4510] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4511] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4512] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4513] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4514] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

In [69]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4519_4529.csv", 
        start_idx=4520,
        end_idx=4529
    )

--- Gentle Session Started ---
Targeting: 4520 to 4529
[4528] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4529] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 2
Total Session Time: 25.22 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4529_4539.csv", 
        start_idx=4530,
        end_idx=4539
    )

--- Gentle Session Started ---
Targeting: 4530 to 4539
[4530] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4531] Generating image... (Session: 2)


In [72]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4539_4549.csv", 
        start_idx=4540,
        end_idx=4549
    )

--- Gentle Session Started ---
Targeting: 4540 to 4549
[4540] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4541] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4542] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4548] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4549] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 76.87 minutes


In [73]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4549_4559.csv", 
        start_idx=4550,
        end_idx=4559
    )

--- Gentle Session Started ---
Targeting: 4550 to 4559
[4550] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4551] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4552] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4558] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4559] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 63.58 minutes


In [74]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4559_4569.csv", 
        start_idx=4560,
        end_idx=4569
    )

--- Gentle Session Started ---
Targeting: 4560 to 4569
[4560] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4561] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4562] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4568] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4569] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 64.58 minutes


In [75]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4569_4579.csv", 
        start_idx=4570,
        end_idx=4579
    )

--- Gentle Session Started ---
Targeting: 4570 to 4579
[4570] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4571] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4572] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4578] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4579] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 63.93 minutes


In [76]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4579_4589.csv", 
        start_idx=4580,
        end_idx=4589
    )

--- Gentle Session Started ---
Targeting: 4580 to 4589
[4580] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4581] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4582] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4588] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4589] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 64.46 minutes


In [77]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4589_4599.csv", 
        start_idx=4590,
        end_idx=4599
    )

--- Gentle Session Started ---
Targeting: 4590 to 4599
[4590] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4591] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4592] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4598] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4599] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 61.84 minutes


In [78]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4599_4609.csv", 
        start_idx=4600,
        end_idx=4609
    )

 Error loading CSV: [Errno 2] No such file or directory: 'captions_4599_4609.csv'


In [79]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4609_4619.csv", 
        start_idx=4610,
        end_idx=4619
    )

--- Gentle Session Started ---
Targeting: 4610 to 4619
[4610] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4611] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4612] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4618] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4619] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 61.31 minutes


In [80]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4619_4629.csv", 
        start_idx=4620,
        end_idx=4629
    )

--- Gentle Session Started ---
Targeting: 4620 to 4629
[4620] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4621] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4622] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4628] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4629] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 65.21 minutes


In [81]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4629_4639.csv", 
        start_idx=4630,
        end_idx=4639
    )

--- Gentle Session Started ---
Targeting: 4630 to 4639
[4630] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4631] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4632] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4638] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4639] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 65.50 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4639_4649.csv", 
        start_idx=4640,
        end_idx=4649
    )

--- Gentle Session Started ---
Targeting: 4640 to 4649
[4640] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4641] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4642] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4649_4659.csv", 
        start_idx=4650,
        end_idx=4659
    )

--- Gentle Session Started ---
Targeting: 4650 to 4659
[4652] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

In [2]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4659_4669.csv", 
        start_idx=4660,
        end_idx=4669
    )

--- Gentle Session Started ---
Targeting: 4660 to 4669

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.01 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4669_4679.csv", 
        start_idx=4670,
        end_idx=4679
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4679_4689.csv", 
        start_idx=4680,
        end_idx=4689
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4689_4699.csv", 
        start_idx=4690,
        end_idx=4699
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4699_4709.csv", 
        start_idx=4700,
        end_idx=4709
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4709_4719.csv", 
        start_idx=4710,
        end_idx=4719
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4719_4729.csv", 
        start_idx=4720,
        end_idx=4729
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4729_4739.csv", 
        start_idx=4730,
        end_idx=4739
    )

In [3]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4739_4749.csv", 
        start_idx=4740,
        end_idx=4749
    )

--- Gentle Session Started ---
Targeting: 4740 to 4749
[4748] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4749] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 2
Total Session Time: 11.69 minutes


In [4]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4749_4759.csv", 
        start_idx=4750,
        end_idx=4759
    )

--- Gentle Session Started ---
Targeting: 4750 to 4759
[4750] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4751] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4752] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4758] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4759] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 57.58 minutes


In [5]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4759_4769.csv", 
        start_idx=4760,
        end_idx=4769
    )

--- Gentle Session Started ---
Targeting: 4760 to 4769
[4760] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4761] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4762] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4768] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4769] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 54.35 minutes


In [6]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4769_4779.csv", 
        start_idx=4770,
        end_idx=4779
    )

--- Gentle Session Started ---
Targeting: 4770 to 4779
[4770] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4771] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4772] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4778] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4779] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 53.50 minutes


In [7]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4779_4789.csv", 
        start_idx=4780,
        end_idx=4789
    )

--- Gentle Session Started ---
Targeting: 4780 to 4789
[4780] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4781] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4782] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4788] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4789] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 54.01 minutes


In [8]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4789_4799.csv", 
        start_idx=4790,
        end_idx=4799
    )

--- Gentle Session Started ---
Targeting: 4790 to 4799
[4790] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4791] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4792] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4798] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4799] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 54.16 minutes


In [9]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4799_4809.csv", 
        start_idx=4800,
        end_idx=4809
    )

--- Gentle Session Started ---
Targeting: 4800 to 4809
[4800] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4801] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4802] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4808] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4809] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 55.27 minutes


In [10]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4809_4819.csv", 
        start_idx=4810,
        end_idx=4819
    )

--- Gentle Session Started ---
Targeting: 4810 to 4819
[4810] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4811] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4812] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4818] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4819] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 55.09 minutes


In [11]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4819_4829.csv", 
        start_idx=4820,
        end_idx=4829
    )

--- Gentle Session Started ---
Targeting: 4820 to 4829
[4820] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4821] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4822] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4828] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4829] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 55.16 minutes


In [12]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4829_4839.csv", 
        start_idx=4830,
        end_idx=4839
    )

--- Gentle Session Started ---
Targeting: 4830 to 4839
[4830] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4831] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4832] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4838] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4839] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 55.29 minutes


In [13]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4839_4849.csv", 
        start_idx=4840,
        end_idx=4849
    )

--- Gentle Session Started ---
Targeting: 4840 to 4849
[4840] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4841] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4842] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4848] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4849] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 55.67 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4849_4859.csv", 
        start_idx=4850,
        end_idx=4859
    )

--- Gentle Session Started ---
Targeting: 4850 to 4859
[4850] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4851] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4852] Generating image... (Session: 3)


...

  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4856] Generating image... (Session: 7)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4857] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4859_4869.csv", 
        start_idx=4860,
        end_idx=4869
    )

In [17]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4869_4879.csv", 
        start_idx=4870,
        end_idx=4879
    )

--- Gentle Session Started ---
Targeting: 4870 to 4879
[4878] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4879] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 2
Total Session Time: 12.29 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4879_4889.csv", 
        start_idx=4880,
        end_idx=4889
    )

--- Gentle Session Started ---
Targeting: 4880 to 4889
[4880] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4889_4899.csv", 
        start_idx=4890,
        end_idx=4899
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4899_4909.csv", 
        start_idx=4900,
        end_idx=4909
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4909_4919.csv", 
        start_idx=4910,
        end_idx=4919
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4919_4929.csv", 
        start_idx=4920,
        end_idx=4929
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4929_4939.csv", 
        start_idx=4930,
        end_idx=4939
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4939_4949.csv", 
        start_idx=4940,
        end_idx=4949
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4949_4959.csv", 
        start_idx=4950,
        end_idx=4959
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4959_4969.csv", 
        start_idx=4960,
        end_idx=4969
    )

In [27]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4969_4979.csv", 
        start_idx=4970,
        end_idx=4979
    )

--- Gentle Session Started ---
Targeting: 4970 to 4979
[4979] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.20 minutes


In [28]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4979_4989.csv", 
        start_idx=4980,
        end_idx=4989
    )

--- Gentle Session Started ---
Targeting: 4980 to 4989
[4980] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4981] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4982] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[4988] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4989] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 55.90 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4989_4999.csv", 
        start_idx=4990,
        end_idx=4999
    )

--- Gentle Session Started ---
Targeting: 4990 to 4999
[4990] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4991] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4992] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4993] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4994] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

## Error Handling

In [31]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4861_4861.csv", 
        start_idx=4861,
        end_idx=4861
    )

--- Gentle Session Started ---
Targeting: 4861 to 4861
[4861] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.63 minutes


In [32]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4741_4741.csv", 
        start_idx=4741,
        end_idx=4741
    )

--- Gentle Session Started ---
Targeting: 4741 to 4741
[4741] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.46 minutes


In [33]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4420_4420.csv", 
        start_idx=4420,
        end_idx=4420
    )

--- Gentle Session Started ---
Targeting: 4420 to 4420
[4420] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.94 minutes


In [34]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4393_4393.csv", 
        start_idx=4393,
        end_idx=4393
    )

--- Gentle Session Started ---
Targeting: 4393 to 4393
[4393] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 8.50 minutes


In [37]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4293_4293.csv", 
        start_idx=4293,
        end_idx=4293
    )

--- Gentle Session Started ---
Targeting: 4293 to 4293
[4293] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.30 minutes


In [38]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4234_4234.csv", 
        start_idx=4234,
        end_idx=4234
    )

--- Gentle Session Started ---
Targeting: 4234 to 4234
[4234] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.35 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4144_4147.csv", 
        start_idx=4144,
        end_idx=4147
    )

--- Gentle Session Started ---
Targeting: 4144 to 4147
[4144] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4145] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[4146] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4193_4193.csv", 
        start_idx=4193,
        end_idx=4193
    )

In [44]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4090_4090.csv", 
        start_idx=4090,
        end_idx=4090
    )

--- Gentle Session Started ---
Targeting: 4090 to 4090
[4090] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 16.85 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3781_3781.csv", 
        start_idx=3781,
        end_idx=3781
    )

--- Gentle Session Started ---
Targeting: 3781 to 3781
[3781] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...


In [52]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3424_3424.csv", 
        start_idx=3424,
        end_idx=3424
    )

--- Gentle Session Started ---
Targeting: 3424 to 3424
[3424] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.43 minutes


In [55]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3409_3409.csv", 
        start_idx=3409,
        end_idx=3409
    )

--- Gentle Session Started ---
Targeting: 3409 to 3409
[3409] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.21 minutes


In [3]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3400_3408.csv", 
        start_idx=3400,
        end_idx=3408
    )

--- Gentle Session Started ---
Targeting: 3400 to 3408
[3408] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 17.18 minutes


In [4]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3391_3391.csv", 
        start_idx=3391,
        end_idx=3391
    )

--- Gentle Session Started ---
Targeting: 3391 to 3391
[3391] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 8.28 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_0091_0091.csv", 
        start_idx=91,
        end_idx=91
    )

--- Gentle Session Started ---
Targeting: 91 to 91
[91] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

In [6]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_0091_0091.csv", 
        start_idx=91,
        end_idx=91
    )

--- Gentle Session Started ---
Targeting: 91 to 91

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [7]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_0111_0111.csv", 
        start_idx=111,
        end_idx=111
    )

--- Gentle Session Started ---
Targeting: 111 to 111
[111] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.22 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_0111_0111.csv", 
        start_idx=111,
        end_idx=111
    )

In [10]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_0220_0220.csv", 
        start_idx=220,
        end_idx=220
    )

--- Gentle Session Started ---
Targeting: 220 to 220
[220] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 4.68 minutes


In [9]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_0530_0530.csv", 
        start_idx=530,
        end_idx=530
    )

--- Gentle Session Started ---
Targeting: 530 to 530
[530] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.16 minutes


In [12]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1022_1022.csv", 
        start_idx=1022,
        end_idx=1022
    )

--- Gentle Session Started ---
Targeting: 1022 to 1022
[1022] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.12 minutes


In [24]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1355_1355.csv", 
        start_idx=1355,
        end_idx=1355
    )

--- Gentle Session Started ---
Targeting: 1355 to 1355
[1355] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 4.73 minutes


In [23]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1356_1356.csv", 
        start_idx=1356,
        end_idx=1356
    )

--- Gentle Session Started ---
Targeting: 1356 to 1356
[1356] Generating image... (Session: 1)


KeyboardInterrupt: 

In [25]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1358_1358.csv", 
        start_idx=1358,
        end_idx=1358
    )

--- Gentle Session Started ---
Targeting: 1358 to 1358
[1358] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.02 minutes


In [26]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1623_1623.csv", 
        start_idx=1623,
        end_idx=1623
    )

--- Gentle Session Started ---
Targeting: 1623 to 1623
[1623] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.62 minutes


In [27]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1626_1626.csv", 
        start_idx=1626,
        end_idx=1626
    )

--- Gentle Session Started ---
Targeting: 1626 to 1626
[1626] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 4.68 minutes


In [28]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1632_1633.csv", 
        start_idx=1632,
        end_idx=1633
    )

--- Gentle Session Started ---
Targeting: 1632 to 1633
[1632] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1633] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 2
Total Session Time: 10.40 minutes


In [29]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1700_1700.csv", 
        start_idx=1700,
        end_idx=1700
    )

--- Gentle Session Started ---
Targeting: 1700 to 1700
[1700] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.83 minutes


In [30]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1749_1749.csv", 
        start_idx=1749,
        end_idx=1749
    )

--- Gentle Session Started ---
Targeting: 1749 to 1749
[1749] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.52 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2181_2181.csv", 
        start_idx=2181,
        end_idx=2181
    )

In [35]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2397_2397.csv", 
        start_idx=2397,
        end_idx=2397
    )

--- Gentle Session Started ---
Targeting: 2397 to 2397
[2397] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.49 minutes


In [33]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3243_3245.csv", 
        start_idx=3243,
        end_idx=3245
    )

--- Gentle Session Started ---
Targeting: 3243 to 3245
[3243] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.25 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1660_1670.csv", 
        start_idx=1660,
        end_idx=1670
    )

--- Gentle Session Started ---
Targeting: 1660 to 1670
[1660] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1661] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1662] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...


In [37]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1950_1959.csv", 
        start_idx=1950,
        end_idx=1958
    )

--- Gentle Session Started ---
Targeting: 1950 to 1958
[1950] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1951] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1952] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[1957] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1958] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 50.18 minutes


In [40]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1942_1949.csv", 
        start_idx=1942,
        end_idx=1949
    )

--- Gentle Session Started ---
Targeting: 1942 to 1949
[1942] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1943] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1944] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[1948] Generating image... (Session: 7)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1949] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 8
Total Session Time: 43.09 minutes


In [47]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2116_2116.csv", 
        start_idx=2116,
        end_idx=2116
    )

--- Gentle Session Started ---
Targeting: 2116 to 2116
[2116] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 17.13 minutes


In [51]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2111_2111.csv", 
        start_idx=2111,
        end_idx=2111
    )

--- Gentle Session Started ---
Targeting: 2111 to 2111
[2111] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 10.97 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2112_2119.csv", 
        start_idx=2112,
        end_idx=2119
    )

--- Gentle Session Started ---
Targeting: 2112 to 2119
[2113] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

In [55]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2151_2159.csv", 
        start_idx=2151,
        end_idx=2159
    )

--- Gentle Session Started ---
Targeting: 2151 to 2159
[2159] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.64 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2181_2181.csv", 
        start_idx=2181,
        end_idx=2181
    )

--- Gentle Session Started ---
Targeting: 2181 to 2181
[2181] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2200_2209.csv", 
        start_idx=2200,
        end_idx=2209
    )

In [56]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1356_1356.csv", 
        start_idx=1356,
        end_idx=1356
    )

--- Gentle Session Started ---
Targeting: 1356 to 1356
[1356] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 4.93 minutes


In [59]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2301_2310.csv", 
        start_idx=2301,
        end_idx=2310
    )

--- Gentle Session Started ---
Targeting: 2301 to 2310
[2301] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2302] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2303] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2309] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2310] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 52.83 minutes


In [61]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2301_2310.csv", 
        start_idx=2301,
        end_idx=2310
    )

--- Gentle Session Started ---
Targeting: 2301 to 2310

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [60]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2311_2319.csv", 
        start_idx=2311,
        end_idx=2319
    )

--- Gentle Session Started ---
Targeting: 2311 to 2319
[2311] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2312] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2313] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2318] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2319] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 47.97 minutes


In [62]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2341_2349.csv", 
        start_idx=2341,
        end_idx=2349
    )

--- Gentle Session Started ---
Targeting: 2341 to 2349
[2341] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2343] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2344] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2348] Generating image... (Session: 7)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2349] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 8
Total Session Time: 43.02 minutes


In [63]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2350_2359.csv", 
        start_idx=2350,
        end_idx=2359
    )

--- Gentle Session Started ---
Targeting: 2350 to 2359
[2350] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2351] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2352] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2358] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2359] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 53.52 minutes


In [64]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2430_2439.csv", 
        start_idx=2430,
        end_idx=2439
    )

--- Gentle Session Started ---
Targeting: 2430 to 2439
[2430] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2431] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2432] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2438] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2439] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 52.51 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2430_2439.csv", 
        start_idx=2430,
        end_idx=2439
    )

In [65]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2440_2449.csv", 
        start_idx=2440,
        end_idx=2449
    )

--- Gentle Session Started ---
Targeting: 2440 to 2449
[2440] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2441] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2442] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2448] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2449] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 47.57 minutes


In [67]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2452_2452.csv", 
        start_idx=2452,
        end_idx=2452
    )

--- Gentle Session Started ---
Targeting: 2452 to 2452
[2452] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.36 minutes


In [68]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2462_2462.csv", 
        start_idx=2462,
        end_idx=2462
    )

--- Gentle Session Started ---
Targeting: 2462 to 2462
[2462] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.54 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2502_2502.csv", 
        start_idx=2502,
        end_idx=2502
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2521_2521.csv", 
        start_idx=2521,
        end_idx=2521
    )

In [69]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2591_2599.csv", 
        start_idx=2591,
        end_idx=2599
    )

--- Gentle Session Started ---
Targeting: 2591 to 2599
[2591] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2592] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2593] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2598] Generating image... (Session: 8)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2599] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 9
Total Session Time: 47.45 minutes


In [70]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2600_2609.csv", 
        start_idx=2600,
        end_idx=2609
    )

--- Gentle Session Started ---
Targeting: 2600 to 2609
[2600] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2601] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2602] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2608] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2609] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 54.87 minutes


In [71]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2824_2824.csv", 
        start_idx=2824,
        end_idx=2824
    )

--- Gentle Session Started ---
Targeting: 2824 to 2824
[2824] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 4.93 minutes


In [72]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2827_2827.csv", 
        start_idx=2827,
        end_idx=2827
    )

--- Gentle Session Started ---
Targeting: 2827 to 2827
[2827] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 5.54 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2831_2835.csv", 
        start_idx=2831,
        end_idx=2835
    )

In [73]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2836_2840.csv", 
        start_idx=2836,
        end_idx=2840
    )

--- Gentle Session Started ---
Targeting: 2836 to 2840
[2836] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2838] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2839] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2840] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 4
Total Session Time: 22.62 minutes


In [74]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2841_2845.csv", 
        start_idx=2841,
        end_idx=2845
    )

--- Gentle Session Started ---
Targeting: 2841 to 2845
[2841] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2842] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2843] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2844] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2845] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 26.50 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2846_2849.csv", 
        start_idx=2846,
        end_idx=2849
    )

--- Gentle Session Started ---
Targeting: 2846 to 2849
[2846] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...


In [76]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2200_2209.csv", 
        start_idx=2209,
        end_idx=2209
    )

--- Gentle Session Started ---
Targeting: 2209 to 2209
[2209] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 18.24 minutes


In [77]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2305_2305.csv", 
        start_idx=2305,
        end_idx=2305
    )

--- Gentle Session Started ---
Targeting: 2305 to 2305
[2305] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 20.44 minutes


In [78]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2315_2315.csv", 
        start_idx=2315,
        end_idx=2315
    )

--- Gentle Session Started ---
Targeting: 2315 to 2315
[2315] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.96 minutes


In [80]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_971_971.csv", 
        start_idx=971,
        end_idx=971
    )

--- Gentle Session Started ---
Targeting: 971 to 971
[971] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 16.36 minutes


In [81]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1042_1042.csv", 
        start_idx=1042,
        end_idx=1042
    )

--- Gentle Session Started ---
Targeting: 1042 to 1042
[1042] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 20.46 minutes


In [82]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1625_1625.csv", 
        start_idx=1625,
        end_idx=1625
    )

--- Gentle Session Started ---
Targeting: 1625 to 1625
[1625] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.61 minutes


In [79]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2341_2349.csv", 
        start_idx=2342,
        end_idx=2342
    )

--- Gentle Session Started ---
Targeting: 2342 to 2342
[2342] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.60 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2440_2445.csv", 
        start_idx=2440,
        end_idx=2445
    )

--- Gentle Session Started ---
Targeting: 2440 to 2445
[2440] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2441] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

In [85]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2446_2449.csv", 
        start_idx=2446,
        end_idx=2449
    )

--- Gentle Session Started ---
Targeting: 2446 to 2449
[2446] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2449] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 2
Total Session Time: 19.01 minutes


In [86]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2502_2502.csv", 
        start_idx=2502,
        end_idx=2502
    )

--- Gentle Session Started ---
Targeting: 2502 to 2502
[2502] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 8.09 minutes


In [87]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2521_2521.csv", 
        start_idx=2521,
        end_idx=2521
    )

--- Gentle Session Started ---
Targeting: 2521 to 2521
[2521] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 21.44 minutes


In [88]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2831_2835.csv", 
        start_idx=2831,
        end_idx=2835
    )

--- Gentle Session Started ---
Targeting: 2831 to 2835
[2831] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2832] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2833] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[2834] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2835] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 87.36 minutes


In [108]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2836_2840.csv", 
        start_idx=2836,
        end_idx=2840
    )

--- Gentle Session Started ---
Targeting: 2836 to 2840

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.01 minutes


In [155]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2847_2847.csv", 
        start_idx=2847,
        end_idx=2847
    )

--- Gentle Session Started ---
Targeting: 2847 to 2847

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [156]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2846_2849.csv", 
        start_idx=2848,
        end_idx=2848
    )

--- Gentle Session Started ---
Targeting: 2848 to 2848

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2846_2849.csv", 
        start_idx=2848,
        end_idx=2848
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2928_2928.csv", 
        start_idx=2928,
        end_idx=2928
    )

In [157]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2909_2909.csv", 
        start_idx=2909,
        end_idx=2909
    )

--- Gentle Session Started ---
Targeting: 2909 to 2909

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [109]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2929_2929.csv", 
        start_idx=2929,
        end_idx=2929
    )

--- Gentle Session Started ---
Targeting: 2929 to 2929
[2929] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.33 minutes


In [110]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3011_3015.csv", 
        start_idx=3011,
        end_idx=3015
    )

--- Gentle Session Started ---
Targeting: 3011 to 3015

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [111]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3016_3020.csv", 
        start_idx=3016,
        end_idx=3020
    )

--- Gentle Session Started ---
Targeting: 3016 to 3020

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [112]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3021_3022.csv", 
        start_idx=3022,
        end_idx=3022
    )

--- Gentle Session Started ---
Targeting: 3022 to 3022

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.01 minutes


In [158]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3042_3046.csv", 
        start_idx=3042,
        end_idx=3046
    )

--- Gentle Session Started ---
Targeting: 3042 to 3046

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [114]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3046_3049.csv", 
        start_idx=3047,
        end_idx=3049
    )

--- Gentle Session Started ---
Targeting: 3047 to 3049

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [115]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3073_3073.csv", 
        start_idx=3073,
        end_idx=3073
    )

--- Gentle Session Started ---
Targeting: 3073 to 3073

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [116]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3070_3075.csv", 
        start_idx=3070,
        end_idx=3075
    )

--- Gentle Session Started ---
Targeting: 3070 to 3075

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [117]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3076_3079.csv", 
        start_idx=3076,
        end_idx=3079
    )

--- Gentle Session Started ---
Targeting: 3076 to 3079

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [118]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3181_3185.csv", 
        start_idx=3181,
        end_idx=3185
    )

--- Gentle Session Started ---
Targeting: 3181 to 3185

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [119]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3181_3185.csv", 
        start_idx=3181,
        end_idx=3185
    )

--- Gentle Session Started ---
Targeting: 3181 to 3185

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [120]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3186_3189.csv", 
        start_idx=3186,
        end_idx=3189
    )

--- Gentle Session Started ---
Targeting: 3186 to 3189
[3186] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3187] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3188] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3189] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 4
Total Session Time: 26.03 minutes


In [121]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3212_3212.csv", 
        start_idx=3212,
        end_idx=3212
    )

--- Gentle Session Started ---
Targeting: 3212 to 3212
[3212] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.06 minutes


In [122]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3210_3215.csv", 
        start_idx=3210,
        end_idx=3215
    )

--- Gentle Session Started ---
Targeting: 3210 to 3215
[3210] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3211] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3213] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3214] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3215] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 33.38 minutes


In [123]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3216_3219.csv", 
        start_idx=3216,
        end_idx=3219
    )

--- Gentle Session Started ---
Targeting: 3216 to 3219
[3216] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3217] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3218] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3219] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 4
Total Session Time: 25.70 minutes


In [124]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3220_3224.csv", 
        start_idx=3220,
        end_idx=3224
    )

--- Gentle Session Started ---
Targeting: 3220 to 3224
[3220] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3221] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3222] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3223] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3224] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 32.54 minutes


In [125]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3225_3229.csv", 
        start_idx=3225,
        end_idx=3229
    )

--- Gentle Session Started ---
Targeting: 3225 to 3229
[3225] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3226] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3227] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3228] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3229] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 31.76 minutes


In [126]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3230_3234.csv", 
        start_idx=3230,
        end_idx=3234
    )

--- Gentle Session Started ---
Targeting: 3230 to 3234
[3230] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3231] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3232] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3233] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3234] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 31.38 minutes


In [127]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3235_3239.csv", 
        start_idx=3235,
        end_idx=3239
    )

--- Gentle Session Started ---
Targeting: 3235 to 3239
[3235] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3236] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3237] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3238] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3239] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 32.18 minutes


In [128]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3244_3245.csv", 
        start_idx=3244,
        end_idx=3245
    )

--- Gentle Session Started ---
Targeting: 3244 to 3245
[3244] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3245] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 2
Total Session Time: 13.23 minutes


In [129]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3244_3245.csv", 
        start_idx=3244,
        end_idx=3245
    )

--- Gentle Session Started ---
Targeting: 3244 to 3245

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [130]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3391_3391.csv", 
        start_idx=3391,
        end_idx=3391
    )

--- Gentle Session Started ---
Targeting: 3391 to 3391

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [131]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3393_3397.csv", 
        start_idx=3393,
        end_idx=3397
    )

--- Gentle Session Started ---
Targeting: 3393 to 3397
[3393] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3394] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3395] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3396] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3397] Generating image... (Session: 5)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 5
Total Session Time: 33.40 minutes


In [132]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3398_3399.csv", 
        start_idx=3398,
        end_idx=3399
    )

--- Gentle Session Started ---
Targeting: 3398 to 3399
[3398] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3399] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 2
Total Session Time: 12.72 minutes


In [133]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3240_3242.csv", 
        start_idx=3240,
        end_idx=3242
    )

--- Gentle Session Started ---
Targeting: 3240 to 3242
[3240] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3241] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3242] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 3
Total Session Time: 53.13 minutes


In [136]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3016_3020.csv", 
        start_idx=3019,
        end_idx=3019
    )

--- Gentle Session Started ---
Targeting: 3019 to 3019
[3019] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 17.34 minutes


In [159]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3016_3020.csv", 
        start_idx=3019,
        end_idx=3019
    )

--- Gentle Session Started ---
Targeting: 3019 to 3019

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [138]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_140_150.csv", 
        start_idx=140,
        end_idx=150
    )

--- Gentle Session Started ---
Targeting: 140 to 150
[148] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[149] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 2
Total Session Time: 14.11 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1079_1089.csv", 
        start_idx=1080,
        end_idx=1089
    )

--- Gentle Session Started ---
Targeting: 1080 to 1089
[1081] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1082] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[1083] Generating image... (Session: 3)


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1089_1099.csv", 
        start_idx=1090,
        end_idx=1099
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1339_1349.csv", 
        start_idx=1340,
        end_idx=1349
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1464_1464.csv", 
        start_idx=1464,
        end_idx=1464
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1539_1549.csv", 
        start_idx=1540,
        end_idx=1549
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1769_1779.csv", 
        start_idx=1770,
        end_idx=1779
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1779_1789.csv", 
        start_idx=1780,
        end_idx=1789
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1789_1799.csv", 
        start_idx=1790,
        end_idx=1799
    )

In [161]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1839_1849.csv", 
        start_idx=1840,
        end_idx=1849
    )

--- Gentle Session Started ---
Targeting: 1840 to 1849

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [162]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1941_1941.csv", 
        start_idx=1941,
        end_idx=1941
    )

--- Gentle Session Started ---
Targeting: 1941 to 1941

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [163]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1989_1999.csv", 
        start_idx=1990,
        end_idx=1999
    )

--- Gentle Session Started ---
Targeting: 1990 to 1999
[1997] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 17.88 minutes


In [164]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2019_2029.csv", 
        start_idx=2020,
        end_idx=2029
    )

--- Gentle Session Started ---
Targeting: 2020 to 2029

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [154]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2419_2429.csv", 
        start_idx=2420,
        end_idx=2429
    )

--- Gentle Session Started ---
Targeting: 2420 to 2429
[2429] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 7.03 minutes


In [167]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2196_2196.csv", 
        start_idx=2196,
        end_idx=2196
    )

--- Gentle Session Started ---
Targeting: 2196 to 2196

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [168]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2739_2749.csv", 
        start_idx=2740,
        end_idx=2749
    )

--- Gentle Session Started ---
Targeting: 2740 to 2749
[2746] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2747] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2748] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2749] Generating image... (Session: 4)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 4
Total Session Time: 31.54 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2849_2859.csv", 
        start_idx=2850,
        end_idx=2859
    )

--- Gentle Session Started ---
Targeting: 2850 to 2859
[2856] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[2857] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

## Here

In [4]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_2909_2919.csv", 
        start_idx=2910,
        end_idx=2919
    )

--- Gentle Session Started ---
Targeting: 2910 to 2919
[2919] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 8.53 minutes


In [3]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3129_3139.csv", 
        start_idx=3130,
        end_idx=3139
    )

--- Gentle Session Started ---
Targeting: 3130 to 3139
[3137] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3138] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3139] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 3
Total Session Time: 36.10 minutes


In [6]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3259_3269.csv", 
        start_idx=3260,
        end_idx=3269
    )

--- Gentle Session Started ---
Targeting: 3260 to 3269
[3260] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3261] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3262] Generating image... (Session: 3)


...

Short 15s rest to keep hardware cool...
[3268] Generating image... (Session: 9)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3269] Generating image... (Session: 10)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 10
Total Session Time: 65.13 minutes


In [8]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3289_3299.csv", 
        start_idx=3299,
        end_idx=3299
    )

--- Gentle Session Started ---
Targeting: 3299 to 3299
[3299] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 6.51 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3639_3649.csv", 
        start_idx=3640,
        end_idx=3649
    )

--- Gentle Session Started ---
Targeting: 3640 to 3649
[3643] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3644] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3739_3749.csv", 
        start_idx=3740,
        end_idx=3749
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3809_3819.csv", 
        start_idx=3819,
        end_idx=3819
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_3979_3989.csv", 
        start_idx=3980,
        end_idx=3989
    )

In [11]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4009_4019.csv", 
        start_idx=4019,
        end_idx=4019
    )

--- Gentle Session Started ---
Targeting: 4019 to 4019
[4019] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 17.09 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4019_4029.csv", 
        start_idx=4020,
        end_idx=4029
    )

--- Gentle Session Started ---
Targeting: 4020 to 4029
[4020] Generating image... (Session: 1)


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4599_4609.csv", 
        start_idx=4600,
        end_idx=4609
    )

In [15]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_4909_4919.csv", 
        start_idx=4910,
        end_idx=4919
    )

--- Gentle Session Started ---
Targeting: 4910 to 4919
[4919] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 16.82 minutes


In [16]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1997_1997.csv", 
        start_idx=1997,
        end_idx=1997
    )

--- Gentle Session Started ---
Targeting: 1997 to 1997
[1997] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 16.80 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_150_159.csv", 
        start_idx=150,
        end_idx=159
    )

--- Gentle Session Started ---
Targeting: 150 to 159
[150] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[151] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1949_1959.csv", 
        start_idx=1959,
        end_idx=1959
    )

--- Gentle Session Started ---
Targeting: 1959 to 1959
[1959] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

In [14]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=150,
        end_idx=150
    )

--- Gentle Session Started ---
Targeting: 150 to 150

--- Generation Summary ---
New Images Created: 0
Total Session Time: 0.00 minutes


In [15]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=919,
        end_idx=919
    )

--- Gentle Session Started ---
Targeting: 919 to 919
[919] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 20.46 minutes


In [17]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=1159,
        end_idx=1159
    )

--- Gentle Session Started ---
Targeting: 1159 to 1159
[1159] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 18.66 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=3740,
        end_idx=3749
    )

--- Gentle Session Started ---
Targeting: 3740 to 3749
[3742] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3743] Generating image... (Session: 2)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...
[3744] Generating image... (Session: 3)


  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=3819,
        end_idx=3819
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=3890,
        end_idx=3898
    )

In [13]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=3909,
        end_idx=3909
    )

--- Gentle Session Started ---
Targeting: 3909 to 3909
[3909] Generating image... (Session: 1)


  0%|          | 0/8 [00:00<?, ?it/s]

Short 15s rest to keep hardware cool...

--- Generation Summary ---
New Images Created: 1
Total Session Time: 79.53 minutes


In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=3980,
        end_idx=3989
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=4141,
        end_idx=4143
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=4148,
        end_idx=4159
    )

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import time
import gc

def generate_images_range(pipe, csv_path, output_dir="generated", start_idx=1639, end_idx=1649):
    start_time = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=['gemini_caption'])
    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    print(f"--- Gentle Session Started ---")
    print(f"Targeting: {start_idx} to {end_idx}")
    
    generated_count = 0
    session_batch_count = 0 

    for i, row in enumerate(df.itertuples()):
        current_num = start_idx + i
        if current_num > end_idx:
            break

        filename = output_dir / f"flux_generated_{current_num}.png"

        # Skip if exists (to avoid unnecessary work)
        if filename.exists():
            continue

        # --- TIER 2: LONG BATCH REST (Every 10 images) ---
        if session_batch_count > 0 and session_batch_count % 10 == 0:
            print(f"\n 10 images reached. Cooling down for 10 minutes...")
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(600) 
            print(" Rest over. Resuming...\n")

        try:
            caption = str(row.gemini_caption).strip()
            if not caption or caption.lower() == "nan":
                continue

            print(f"[{current_num}] Generating image... (Session: {session_batch_count + 1})")

            with torch.inference_mode():
                image = pipe(
                    prompt=caption,
                    num_inference_steps=8, 
                    guidance_scale=1.0,
                    height=512,
                    width=512
                ).images[0]
            
            image.save(filename)
            generated_count += 1
            session_batch_count += 1 
            
            # --- TIER 1: MICRO-REST (After every single image) ---
            # Gives the fans a chance to catch up and the GPU clock to down-cycle briefly
            print(f"Short 15s rest to keep hardware cool...")
            torch.cuda.empty_cache()
            gc.collect() 
            time.sleep(30) 

        except Exception as e:
            print(f"!! Error at {current_num}: {e}")
            time.sleep(5)
            continue

    duration = (time.time() - start_time) / 60
    print(f"\n--- Generation Summary ---")
    print(f"New Images Created: {generated_count}")
    print(f"Total Session Time: {duration:.2f} minutes")

if __name__ == "__main__":
    generate_images_range(
        pipe=pipe, 
        csv_path="captions_1_4999_final.csv", 
        start_idx=4339,
        end_idx=4339
    )